# transformer_sequence_adapter_batch_v1

验证第三批 sequence/deep adapter：`hawkes_gru_mse`、`mlp_mixer_mse`、`dlinear_mse`、`patch_tst_mse`、`autoformer_mse`、`fedformer_mse`。本轮目标是把事件衰减、Mixer、线性分解、patch、decomposition attention、frequency decomposition 六类 proxy 接入统一 `src/model` 训练、预测、artifact 和 matrix analysis 链路。

## 实验治理

- Baseline：上一轮完成 ALSTM/SFM/DA-RNN/TCN proxy，本轮扩展到 Hawkes/Mixer/DLinear/PatchTST/Autoformer/FEDformer family。
- Direct comparison：事件衰减、Mixer、decomposition、patch、frequency proxy 是否输出不同预测并保持同一 artifact/manifest 契约。
- Role：diagnose，推进 QuantBench sequence/Transformer family 从 smoke fallback 迁移到可训练 adapter。
- Target window：2026-01-01 到 2026-06-09，2026-04-30 前训练，之后测试。
- Changed layer：model。
- External reference：Neural Hawkes、MLP-Mixer、DLinear/LTSF-Linear、PatchTST、Autoformer、FEDformer。当前是可训练 proxy，不宣称完整论文级结构。
- Acceptance：六个模型输出 `paper_strict_adapter`；Mixer/DLinear/Patch/Autoformer/FEDformer/Hawkes 预测路径互不退化为同一个 fallback；PatchTST artifact 保存/加载后保留 adapter metadata。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').exists() and (path / 'AGENTS.md').exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model import fit_model_artifact, load_model_artifact, save_model_artifact
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import fit_predict_model, model_adapter_name, model_adapter_status

CARD = repo_root / 'research' / 'transformer_sequence_adapter_batch_v1'
CARD.mkdir(parents=True, exist_ok=True)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
test = dataset.frame.loc[dataset.frame['date'] > '2026-04-30'].copy()
feature_columns = dataset.feature_columns[:5]
{
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'feature_columns': list(feature_columns),
    'train_end': str(train['date'].max().date()),
    'test_start': str(test['date'].min().date()),
    'test_end': str(test['date'].max().date()),
}


{'train_rows': 82,
 'test_rows': 26,
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d'],
 'train_end': '2026-04-30',
 'test_start': '2026-05-01',
 'test_end': '2026-06-08'}

In [2]:
model_names = (
    'hawkes_gru_mse',
    'mlp_mixer_mse',
    'dlinear_mse',
    'patch_tst_mse',
    'autoformer_mse',
    'fedformer_mse',
)
expected = {
    'hawkes_gru_mse': ('paper_strict_adapter', 'hawkes_gru_event_decay_mse_v1'),
    'mlp_mixer_mse': ('paper_strict_adapter', 'mlp_mixer_sequence_mse_v1'),
    'dlinear_mse': ('paper_strict_adapter', 'dlinear_decomposition_mse_v1'),
    'patch_tst_mse': ('paper_strict_adapter', 'patch_tst_transformer_mse_v1'),
    'autoformer_mse': ('paper_strict_adapter', 'autoformer_decomposition_attention_mse_v1'),
    'fedformer_mse': ('paper_strict_adapter', 'fedformer_frequency_attention_mse_v1'),
}

predictions = pd.concat(
    [fit_predict_model(name, train=train, test=test, feature_columns=feature_columns) for name in model_names],
    ignore_index=True,
)
predictions.to_csv(CARD / 'adapter_predictions.csv', index=False)

for name, (status, adapter_name) in expected.items():
    subset = predictions.loc[predictions['model'] == name]
    assert subset['adapter_status'].unique().tolist() == [status]
    assert subset['model_adapter'].unique().tolist() == [adapter_name]
    assert model_adapter_status(name) == status
    assert model_adapter_name(name) == adapter_name

wide = predictions.pivot(index='date', columns='model', values='prediction').reset_index()
wide['mixer_minus_dlinear'] = wide['mlp_mixer_mse'] - wide['dlinear_mse']
wide['patch_minus_autoformer'] = wide['patch_tst_mse'] - wide['autoformer_mse']
wide['fedformer_minus_dlinear'] = wide['fedformer_mse'] - wide['dlinear_mse']
wide['hawkes_minus_dlinear'] = wide['hawkes_gru_mse'] - wide['dlinear_mse']
wide.to_csv(CARD / 'transformer_sequence_prediction_deltas.csv', index=False)

assert not wide['mlp_mixer_mse'].round(10).equals(wide['dlinear_mse'].round(10))
assert not wide['patch_tst_mse'].round(10).equals(wide['autoformer_mse'].round(10))
assert not wide['fedformer_mse'].round(10).equals(wide['dlinear_mse'].round(10))
assert not wide['hawkes_gru_mse'].round(10).equals(wide['dlinear_mse'].round(10))
wide.head()


model,date,autoformer_mse,dlinear_mse,fedformer_mse,hawkes_gru_mse,mlp_mixer_mse,patch_tst_mse,mixer_minus_dlinear,patch_minus_autoformer,fedformer_minus_dlinear,hawkes_minus_dlinear
0,2026-05-01,-0.010890,-0.010671,-0.023996,-0.027633,0.001510,-0.013036,0.012181,-0.002146,-0.013324,-0.016961
1,2026-05-04,-0.009931,0.000353,-0.020608,-0.020363,-0.005676,-0.008930,-0.006029,0.001001,-0.020961,-0.020717
2,2026-05-05,0.001175,0.041921,0.023655,0.037206,0.033244,0.002298,-0.008677,0.001123,-0.018266,-0.004716
3,2026-05-06,-0.005432,0.039642,0.055235,0.033196,0.040735,-0.004988,0.001093,0.000443,0.015593,-0.006446
4,2026-05-07,0.007764,0.016824,0.037179,0.036429,-0.017196,0.007590,-0.034020,-0.000174,0.020355,0.019605


In [3]:
summary_rows = []
group_columns = ['model', 'objective', 'family', 'adapter_status', 'model_adapter']
for keys, group in predictions.groupby(group_columns, dropna=False):
    summary_rows.append(
        {
            **dict(zip(group_columns, keys)),
            'rows': int(len(group)),
            'prediction_mean': float(group['prediction'].mean()),
            'prediction_std': float(group['prediction'].std(ddof=0)),
            'target_corr': float(group['prediction'].corr(group['target_return'])),
        }
    )
adapter_summary = pd.DataFrame(summary_rows)
adapter_summary.to_csv(CARD / 'adapter_summary.csv', index=False)
adapter_summary


,model,objective,family,adapter_status,model_adapter,rows,prediction_mean,prediction_std,target_corr
0,autoformer_mse,mse,sequence,paper_strict_adapter,autoformer_decomposition_attention_mse_v1,26,0.026085,0.036459,0.061560
1,dlinear_mse,mse,sequence,paper_strict_adapter,dlinear_decomposition_mse_v1,26,0.016249,0.016090,-0.029561
2,fedformer_mse,mse,sequence,paper_strict_adapter,fedformer_frequency_attention_mse_v1,26,0.029864,0.028939,0.057421
3,hawkes_gru_mse,mse,sequence,paper_strict_adapter,hawkes_gru_event_decay_mse_v1,26,0.040307,0.032150,-0.076896
4,mlp_mixer_mse,mse,sequence,paper_strict_adapter,mlp_mixer_sequence_mse_v1,26,0.040309,0.047770,0.060552
5,patch_tst_mse,mse,sequence,paper_strict_adapter,patch_tst_transformer_mse_v1,26,0.028195,0.036576,0.048272


In [4]:
artifact = fit_model_artifact('patch_tst_mse', train=train, feature_columns=feature_columns)
artifact_path = save_model_artifact(CARD / 'patch_tst_mse_artifact.joblib', artifact)
loaded = load_model_artifact(artifact_path)
artifact_predictions = loaded.predict(test)
artifact_metadata = loaded.metadata_payload()
artifact_predictions.to_csv(CARD / 'patch_tst_artifact_predictions.csv', index=False)
(CARD / 'patch_tst_artifact_metadata.json').write_text(
    json.dumps(artifact_metadata, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

assert loaded.adapter_status == 'paper_strict_adapter'
assert artifact_metadata['model_adapter'] == 'patch_tst_transformer_mse_v1'
assert artifact_predictions['model_adapter'].unique().tolist() == ['patch_tst_transformer_mse_v1']
{
    'artifact_path': str(artifact_path.relative_to(CARD)),
    'metadata': artifact_metadata,
    'prediction_rows': int(len(artifact_predictions)),
}


{'artifact_path': 'patch_tst_mse_artifact.joblib',
 'metadata': {'schema_version': 'model_artifact.v1',
  'model_name': 'patch_tst_mse',
  'feature_columns': ['soxl_return_1d',
   'soxx_return_1d',
   'soxl_return_5d',
   'soxx_return_5d',
   'soxl_soxx_relative_5d'],
  'objective': 'mse',
  'family': 'sequence',
  'adapter_status': 'paper_strict_adapter',
  'model_adapter': 'patch_tst_transformer_mse_v1',
  'target_column': 'target_return',
  'train_rows': 82},
 'prediction_rows': 26}

In [5]:
matrix = run_quantbench_matrix_smoke(
    card_dir=CARD / 'matrix_card',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=model_names,
)
manifest = json.loads((CARD / 'matrix_card' / 'artifact_manifest.json').read_text(encoding='utf-8'))
assert manifest['metadata']['adapter_status'] == 'paper_strict_adapter'
assert manifest['metadata']['adapter_statuses'] == ['paper_strict_adapter']
matrix.model_comparison.to_csv(CARD / 'matrix_model_comparison.csv', index=False)
{
    'matrix_status': matrix.summary['status'],
    'top_model': matrix.summary['top_model'],
    'adapter_status': manifest['metadata']['adapter_status'],
    'model_adapters': manifest['metadata']['model_adapters'],
    'manifest_hash': manifest['manifest_hash'],
}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'matrix_status': 'smoke_completed',
 'top_model': 'mlp_mixer_mse',
 'adapter_status': 'paper_strict_adapter',
 'model_adapters': ['autoformer_decomposition_attention_mse_v1',
  'dlinear_decomposition_mse_v1',
  'fedformer_frequency_attention_mse_v1',
  'hawkes_gru_event_decay_mse_v1',
  'mlp_mixer_sequence_mse_v1',
  'patch_tst_transformer_mse_v1'],
 'manifest_hash': '72e5436c6f296520d2224cdef77e8bc826d0d2b79f41b1b82ba8c92eaaca3a84'}

In [6]:
analysis_summary = {
    'schema_version': 'transformer_sequence_adapter_batch_review.v1',
    'status': 'completed',
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'feature_columns': list(feature_columns),
    'models': list(model_names),
    'adapter_statuses': sorted(predictions['adapter_status'].unique().tolist()),
    'model_adapters': sorted(predictions['model_adapter'].unique().tolist()),
    'mixer_vs_dlinear_abs_mean_diff': float(wide['mixer_minus_dlinear'].abs().mean()),
    'patch_vs_autoformer_abs_mean_diff': float(wide['patch_minus_autoformer'].abs().mean()),
    'fedformer_vs_dlinear_abs_mean_diff': float(wide['fedformer_minus_dlinear'].abs().mean()),
    'hawkes_vs_dlinear_abs_mean_diff': float(wide['hawkes_minus_dlinear'].abs().mean()),
    'adapter_summary_path': 'adapter_summary.csv',
    'prediction_path': 'adapter_predictions.csv',
    'delta_path': 'transformer_sequence_prediction_deltas.csv',
    'artifact_path': 'patch_tst_mse_artifact.joblib',
    'matrix_manifest': 'matrix_card/artifact_manifest.json',
    'matrix_top_model': matrix.summary['top_model'],
    'matrix_manifest_hash': manifest['manifest_hash'],
    'external_references': [
        'https://arxiv.org/abs/1612.09328',
        'https://arxiv.org/abs/2105.01601',
        'https://arxiv.org/abs/2205.13504',
        'https://arxiv.org/abs/2211.14730',
        'https://arxiv.org/abs/2106.13008',
        'https://arxiv.org/abs/2201.12740',
    ],
}
(CARD / 'analysis_summary.json').write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2), encoding='utf-8')
analysis_summary


{'schema_version': 'transformer_sequence_adapter_batch_review.v1',
 'status': 'completed',
 'train_rows': 82,
 'test_rows': 26,
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d'],
 'models': ['hawkes_gru_mse',
  'mlp_mixer_mse',
  'dlinear_mse',
  'patch_tst_mse',
  'autoformer_mse',
  'fedformer_mse'],
 'adapter_statuses': ['paper_strict_adapter'],
 'model_adapters': ['autoformer_decomposition_attention_mse_v1',
  'dlinear_decomposition_mse_v1',
  'fedformer_frequency_attention_mse_v1',
  'hawkes_gru_event_decay_mse_v1',
  'mlp_mixer_sequence_mse_v1',
  'patch_tst_transformer_mse_v1'],
 'mixer_vs_dlinear_abs_mean_diff': 0.03321351809427142,
 'patch_vs_autoformer_abs_mean_diff': 0.004536408716096328,
 'fedformer_vs_dlinear_abs_mean_diff': 0.02157981783294907,
 'hawkes_vs_dlinear_abs_mean_diff': 0.029528082993168097,
 'adapter_summary_path': 'adapter_summary.csv',
 'prediction_path': 'adapter_predictions.csv',
 '

## 中文结论

- 做了什么：新增并验证 `hawkes_gru_mse`、`mlp_mixer_mse`、`dlinear_mse`、`patch_tst_mse`、`autoformer_mse`、`fedformer_mse` 六个 sequence/deep adapter；当前实现是事件衰减、Mixer、线性分解、patch、decomposition+patch、frequency+decomposition 的可训练 proxy。
- 核心指标：六个模型的 adapter 状态、预测均值/波动、预测与目标相关性写入 `adapter_summary.csv`；Mixer vs DLinear、PatchTST vs Autoformer、FEDformer vs DLinear、Hawkes vs DLinear 的平均绝对预测差异写入 `analysis_summary.json`。
- regime 差异：本轮小矩阵通过 `matrix_card` 写出 `regime_metrics.csv`、`window_metrics.csv`、`qlib_risk_metrics.csv` 和 `model_effect_deltas.csv`，证明该批 adapter 能进入已有 qlib/regime 分析链路。
- 是否晋升：否。本轮是 adapter 框架推进，不代表生产策略晋升。
- 风险：当前均为 proxy，PatchTST/Autoformer/FEDformer 还不是完整 Transformer 论文结构；后续要在同一 adapter 名称或 schema 下替换为更严格实现。
- 下一步：进入 graph/network family：`market_graph_mse`、`gcn_mse`、`gat_mse`、`rgcn_mse`、`rsr_mse`、`hypergraph_mse`、`thgnn_mse`、`sthan_mse`、`sthgcn_mse`，并保持同一 artifact/manifest/dashboard/analysis 契约。